In [ ]:
import pandas as pd
import os

from pathlib import Path

In [ ]:
# Supress copy warning.
pd.options.mode.chained_assignment = None

In [ ]:
# Release of interest
rel = 12

In [ ]:
wd = '/YOUR/WORKING/DIR'

### Run COJO on PAR1

In [ ]:
label = 'GP2_EUR_AJ'
region = 'PAR1'

xwas_dir = f'{wd}/working/R{rel}_prep/XWAS'
cojo_dir = f'{xwas_dir}/COJO/{region}'
prefix = f'{region}_COJO'

# GCTA is not on PATH by default -- point this at your own install
gcta = '/PATH/TO/gcta64'

os.makedirs(cojo_dir, exist_ok=True)

In [ ]:
# COJO-format sumstats, PAR1 only (pos <= 2,781,479)
# cols: SNP, A1, A2, freq, b, se, p, N -- taken from the merged METAL/LocusZoom file
! zcat {xwas_dir}/meta/METAL/{label}/locuszoom/merged/{label}_PAR_combined_filtered.tsv.gz \
  | awk 'BEGIN{{OFS="\t"; print "SNP","A1","A2","freq","b","se","p","N"}} \
         NR>1 && $2<=2781479 {{print "PAR1:"$2":"$4":"$5, $15, $16, $17, $21, $22, $23, $35}}' \
  > {cojo_dir}/{region}_stats_for_cojo.txt

! tail -n +2 {cojo_dir}/{region}_stats_for_cojo.txt | cut -f1 > {cojo_dir}/{region}_snps.txt

In [ ]:
# LD reference: EUR PAR file -> PAR1, QC'd samples, sumstat SNPs only, numeric chr code for GCTA
! plink2 --pfile {xwas_dir}/combined/PAR/EUR_combined_merged_chrX_FIXED \
  --chr PAR1 \
  --keep {xwas_dir}/prep_files/combined_EUR_covar_FIXED.txt \
  --extract {cojo_dir}/{region}_snps.txt \
  --maf 0.0001 \
  --output-chr 26 \
  --make-bed \
  --out {cojo_dir}/EUR_{region}_for_COJO

In [ ]:
# COJO stepwise selection
! {gcta} \
  --bfile {cojo_dir}/EUR_{region}_for_COJO \
  --maf 0.001 \
  --cojo-file {cojo_dir}/{region}_stats_for_cojo.txt \
  --cojo-slct \
  --thread-num 8 \
  --out {cojo_dir}/{prefix}

### Load COJO Outputs

In [ ]:
d = Path(cojo_dir)

def read_cojo(path):
    # GCTA writes tab-delimited files, sometimes with trailing tabs; whitespace parsing handles both
    return pd.read_csv(path, sep=r"\s+", na_values=["NA", "nan"])

ma   = read_cojo(d / f"{region}_stats_for_cojo.txt")
jma  = read_cojo(d / f"{prefix}.jma.cojo")
cma  = read_cojo(d / f"{prefix}.cma.cojo")
bad  = read_cojo(d / f"{prefix}.freq.badsnps")

# The LD matrix: first column holds SNP names; trailing tabs create empty columns, so drop them
ldr = pd.read_csv(d / f"{prefix}.ldr.cojo", sep="\t", index_col=0)
ldr = ldr.loc[:, ~ldr.columns.str.startswith("Unnamed")].astype(float)

log_text = (d / f"{prefix}.log").read_text()

for name, df in [("input .ma", ma), ("jma", jma), ("cma", cma), ("badsnps", bad), ("ldr", ldr)]:
    print(f"{name:10s} {df.shape}")

In [ ]:
# Print the lines that carry the important information
keys = ["included", "excluded", "selected", "frequency", "matched", "Warning",
        "Error", "collinear", "lambda", "window", "p-value", "Finally"]
for line in log_text.splitlines():
    if any(k.lower() in line.lower() for k in keys):
        print(line)

In [ ]:
print(bad.head())
print(f"{len(bad)} SNPs excluded for frequency mismatch "
      f"({len(bad)/len(ma):.2%} of input)")

### Independent Signals

In [ ]:
jma_sorted = jma.sort_values("pJ")
display(jma_sorted[["Chr","SNP","bp","refA","freq","freq_geno",
                    "b","se","p","bJ","bJ_se","pJ","LD_r"]])

print(f"\n{len(jma)} independent signal(s) selected")
print(f"{(jma.pJ < 5e-8).sum()} remain genome-wide significant in the joint model")

In [ ]:
display(cma.sort_values("pC").head(20)[["SNP","bp","b","p","bC","bC_se","pC"]])

n_resid = (cma.pC < 5e-8).sum()
print(f"SNPs still genome-wide significant after conditioning: {n_resid}")

print(cma.sort_values("pC")[["SNP","bp","p","pC","b","bC"]].head(15))
print("pC < 1e-5:", (cma.pC < 1e-5).sum(), "| pC < 1e-6:", (cma.pC < 1e-6).sum())

In [ ]:
summary = jma_sorted.assign(
    OR_or_beta_marginal=jma_sorted.b,
    OR_or_beta_joint=jma_sorted.bJ,
    attenuation=(1 - jma_sorted.bJ / jma_sorted.b).round(3),
)[["SNP","bp","refA","freq","b","p","bJ","pJ","attenuation"]]
display(summary)
summary.to_csv(f"{cojo_dir}/{region}_COJO_independent_signals.csv", index=False)

### Check Which Variants COJO Saw

In [ ]:
bim = pd.read_csv(f"{cojo_dir}/EUR_{region}_for_COJO.bim", sep=r"\s+", header=None,
                  names=["chr","SNP","cm","bp","A1","A2"])

ma["matched"] = ma.SNP.isin(cma.SNP) | ma.SNP.isin(jma.SNP)
ma["in_bim"]  = ma.SNP.isin(bim.SNP)

print(ma[["in_bim","matched"]].value_counts())   # not in bim vs. in bim but MAF-filtered

gwas_lead = ma.sort_values("p").iloc[0]
print("GWAS lead SNP:", gwas_lead.SNP, gwas_lead.p, "| matched:", gwas_lead.matched)
print("COJO selected:", jma.SNP.tolist(), jma.p.tolist())

# Significant GWAS SNPs that COJO never saw
print(ma[(ma.p < 5e-8) & ~ma.matched].sort_values("p").head(20))